In [ ]:
!pip install ripser

In [2]:
import os
import glob
import hashlib
import numpy as np
import pandas as pd

from ripser import ripser
from scipy.signal import butter, filtfilt
from scipy.stats import iqr, skew, kurtosis, entropy
from scipy.spatial.distance import pdist

# =============================================================================
# CONFIG
# =============================================================================
ROOT_DIR = r"/content/drive/MyDrive/Colab Notebooks/datasets/Stress-Predict-Dataset/Raw_data"
OUTPUT_DIR = r"/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Stress-Predict-Dataset/BVP-TPV-noise"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXCLUDE_SUBJECTS = {"S17"}

FS_BVP = 64

WINDOW_SEC = 60
STEP_SEC = 60
MAJORITY_RATIO_MIN = 0.95

# ---- BVP filter ----
LOWCUT = 0.5
HIGHCUT = 8.0
FILTER_ORDER = 4

# -----------------------------------------------------------------------------
# Noise settings
# -----------------------------------------------------------------------------
NOISE_LEVELS = {
    "alpha_0.01": 0.01,
    "alpha_0.03": 0.03,
    "alpha_0.05": 0.05,
}
RANDOM_SEED = 42

SELECTED_TPV_INDICES = [22, 9, 4, 2, 0, 28, 8, 19, 14, 26, 11]

TPV_FEATURE_NAMES = [
    "birth_mean", "birth_std", "death_mean", "death_std",
    "lifetime_mean", "lifetime_std", "lifetime_max", "lifetime_min",
    "lifetime_median", "lifetime_iqr", "lifetime_skew", "lifetime_kurtosis",
    "birth_max", "birth_min", "birth_median", "birth_skew", "birth_kurtosis",
    "death_max", "death_min", "death_median", "death_skew", "death_kurtosis",
    "num_lifetimes", "top1_lifetime", "top2_lifetime", "lifetime_max_div_min",
    "ph_entropy", "betti_entropy", "avg_persistence_distance", "gini_index",
    "lifetime_variance", "persistent_image_energy",
]
N_FEATURES = len(TPV_FEATURE_NAMES)

# =============================================================================
# Loaders
# =============================================================================
def load_empatica_signal_csv(file_path, signal_name):
    raw = pd.read_csv(file_path, header=None)
    start_ts = float(raw.iloc[0, 0])
    fs = float(raw.iloc[1, 0])
    values = raw.iloc[2:, 0].astype(float).reset_index(drop=True)
    timestamps = start_ts + np.arange(len(values)) / fs
    df = pd.DataFrame({"timestamp": timestamps, signal_name: values})
    return df, start_ts, fs


def load_tags_csv(file_path, drop_first_tag=True):
    raw = pd.read_csv(file_path, header=None)
    tags = raw.iloc[:, 0].dropna().astype(float).reset_index(drop=True)

    if drop_first_tag:
        if len(tags) <= 1:
            raise ValueError(f"Not enough tags after dropping first tag: n={len(tags)}")
        tags = tags.iloc[1:].reset_index(drop=True)

    return pd.DataFrame({
        "tag_idx": np.arange(len(tags)),
        "timestamp": tags
    })


# =============================================================================
# Protocol intervals
# =============================================================================
def build_protocol_intervals(tags_df):
    timestamps = tags_df["timestamp"].values

    phase_info = [
        ("Baseline", 0),
        ("Stroop", 1),
        ("Rest1", 0),
        ("TSST", 1),
        ("Rest2", 0),
        ("Hyperventilation", 1),
        ("Rest3", 0),
        ("Questionnaire", 0),
    ]

    n_intervals_possible = len(timestamps) - 1
    n_use = min(n_intervals_possible, len(phase_info))
    if n_use <= 0:
        raise ValueError(f"No valid intervals from tags. n_tags={len(timestamps)}")

    rows = []
    for i in range(n_use):
        phase_name, label = phase_info[i]
        rows.append({
            "interval_idx": i,
            "phase": phase_name,
            "start": timestamps[i],
            "end": timestamps[i + 1],
            "label": label
        })

    return pd.DataFrame(rows)


def build_window_table_from_intervals(intervals_df, window_sec=60, step_sec=60, majority_ratio_min=0.95):
    rows = []
    for _, r in intervals_df.iterrows():
        phase = r["phase"]
        label = int(r["label"])
        start = float(r["start"])
        end = float(r["end"])

        t = start
        while t + window_sec <= end:
            major_ratio = 1.0
            if major_ratio >= majority_ratio_min:
                rows.append({
                    "phase": phase,
                    "label": label,
                    "window_start": t,
                    "window_end": t + window_sec,
                    "major_ratio": major_ratio
                })
            t += step_sec

    return pd.DataFrame(rows)


# =============================================================================
# Basic utils
# =============================================================================
def fill_nan_with_median(x):
    x = np.asarray(x, dtype=np.float32).reshape(-1).copy()
    if np.isnan(x).any():
        med = np.nanmedian(x)
        if np.isnan(med):
            med = 0.0
        x = np.nan_to_num(x, nan=med, posinf=med, neginf=med)
    return x


def bandpass_filter(sig, fs=64, low=0.5, high=8.0, order=4):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, sig).astype(np.float32)


def zscore_1d(x):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    mu = float(np.mean(x))
    sd = float(np.std(x))
    if sd < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mu) / (sd + 1e-8)).astype(np.float32)


def safe_skew(x):
    if len(x) > 2 and np.std(x) > 1e-12:
        v = skew(x)
        return float(v) if np.isfinite(v) else 0.0
    return 0.0


def safe_kurtosis(x):
    if len(x) > 3 and np.std(x) > 1e-12:
        v = kurtosis(x)
        return float(v) if np.isfinite(v) else 0.0
    return 0.0


def add_gaussian_noise(sig: np.ndarray, alpha: float, rng: np.random.Generator) -> np.ndarray:
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    if alpha <= 0:
        return sig.copy()

    sig_std = float(np.std(sig))
    if sig_std < 1e-8:
        return sig.copy()

    noise = rng.normal(loc=0.0, scale=alpha * sig_std, size=len(sig)).astype(np.float32)
    return (sig + noise).astype(np.float32)


# =============================================================================
# TPV extraction
# =============================================================================
def extract_tpv_features(sig_1d):
    sig = np.asarray(sig_1d, dtype=np.float32).reshape(-1)
    if len(sig) < 3:
        return np.zeros(N_FEATURES, dtype=np.float32)

    s = float(np.std(sig))
    if s < 1e-8:
        return np.zeros(N_FEATURES, dtype=np.float32)

    sig = (sig - float(np.mean(sig))) / (s + 1e-8)
    X = np.stack([sig[:-1], sig[1:]], axis=1)
    dgms = ripser(X, maxdim=1)["dgms"]
    H1 = dgms[1]

    if H1.size == 0:
        return np.zeros(N_FEATURES, dtype=np.float32)

    births = H1[:, 0]
    deaths = H1[:, 1]
    lifetimes = deaths - births

    mask = np.isfinite(births) & np.isfinite(deaths) & np.isfinite(lifetimes)
    births = births[mask]
    deaths = deaths[mask]
    lifetimes = lifetimes[mask]

    if len(lifetimes) == 0:
        return np.zeros(N_FEATURES, dtype=np.float32)

    lifetimes_sorted = np.sort(lifetimes)
    lifetime_sum = float(np.sum(lifetimes))
    lifetime_ratio = lifetimes / (lifetime_sum + 1e-8)
    ph_entropy = float(-np.sum(lifetime_ratio * np.log(lifetime_ratio + 1e-10)))

    bmin = float(np.min(births))
    bmax = float(np.max(births))
    if bmax - bmin < 1e-8:
        betti_entropy = 0.0
    else:
        hist, _ = np.histogram(births, bins=10, range=(bmin, bmax), density=True)
        betti_entropy = float(entropy(hist + 1e-10))

    avg_persistence_distance = float(np.mean(pdist(lifetimes[:, None]))) if len(lifetimes) > 1 else 0.0

    if np.sum(lifetimes_sorted) > 0 and len(lifetimes_sorted) > 1:
        n = len(lifetimes_sorted)
        gini = (2 * np.sum(np.arange(1, n + 1) * lifetimes_sorted)) / (n * np.sum(lifetimes_sorted)) - (n + 1) / n
        gini_index = float(gini)
    else:
        gini_index = 0.0

    lifetime_variance = float(np.var(lifetimes))
    persistent_image_energy = float(np.sum(lifetimes ** 2))

    feats = [
        float(np.mean(births)),
        float(np.std(births)),
        float(np.mean(deaths)),
        float(np.std(deaths)),
        float(np.mean(lifetimes)),
        float(np.std(lifetimes)),
        float(np.max(lifetimes)),
        float(np.min(lifetimes)),
        float(np.median(lifetimes)),
        float(iqr(lifetimes)),
        safe_skew(lifetimes),
        safe_kurtosis(lifetimes),
        float(np.max(births)),
        float(np.min(births)),
        float(np.median(births)),
        safe_skew(births),
        safe_kurtosis(births),
        float(np.max(deaths)),
        float(np.min(deaths)),
        float(np.median(deaths)),
        safe_skew(deaths),
        safe_kurtosis(deaths),
        float(len(lifetimes)),
        float(lifetimes_sorted[-1]),
        float(lifetimes_sorted[-2]) if len(lifetimes_sorted) > 1 else 0.0,
        float(np.max(lifetimes) / (np.min(lifetimes) + 1e-8)),
        ph_entropy,
        betti_entropy,
        avg_persistence_distance,
        gini_index,
        lifetime_variance,
        persistent_image_energy,
    ]
    return np.asarray(feats, dtype=np.float32)


# =============================================================================
# Main extraction (noise only)
# =============================================================================
def extract_subject_bvp_tpv_with_noise(subject_dir, noise_name, alpha, rng):
    subject_id = os.path.basename(subject_dir)
    if subject_id in EXCLUDE_SUBJECTS:
        return pd.DataFrame()

    bvp_path = os.path.join(subject_dir, "BVP.csv")
    tags_path = os.path.join(subject_dir, f"tags_{subject_id}.csv")

    if not (os.path.exists(bvp_path) and os.path.exists(tags_path)):
        return pd.DataFrame()

    bvp_df, _, fs_bvp = load_empatica_signal_csv(bvp_path, "bvp")
    tags_df = load_tags_csv(tags_path, drop_first_tag=True)

    if len(tags_df) < 2:
        print(f"[WARN] {subject_id}: too few tags after dropping first tag")
        return pd.DataFrame()

    if int(round(fs_bvp)) != FS_BVP:
        print(f"[WARN] {subject_id}: fs mismatch BVP={fs_bvp}")

    intervals_df = build_protocol_intervals(tags_df)
    window_df = build_window_table_from_intervals(
        intervals_df,
        window_sec=WINDOW_SEC,
        step_sec=STEP_SEC,
        majority_ratio_min=MAJORITY_RATIO_MIN
    )

    # raw 전체 BVP에 먼저 noise 적용
    bvp_raw = bvp_df["bvp"].values.astype(np.float32)
    bvp_raw = fill_nan_with_median(bvp_raw)
    bvp_noisy = add_gaussian_noise(bvp_raw, alpha=alpha, rng=rng)

    bvp_time = bvp_df["timestamp"].values.astype(np.float64)

    rows = []
    window_counter = 0

    expected_len = int(WINDOW_SEC * FS_BVP)

    for _, w in window_df.iterrows():
        t0 = float(w["window_start"])
        t1 = float(w["window_end"])

        mask_bvp = (bvp_time >= t0) & (bvp_time < t1)
        seg_bvp = bvp_noisy[mask_bvp]

        if len(seg_bvp) != expected_len:
            continue

        # no-QC TPV pipeline
        seg_bvp = fill_nan_with_median(seg_bvp)
        seg_bvp_bp = bandpass_filter(seg_bvp, fs=FS_BVP, low=LOWCUT, high=HIGHCUT, order=FILTER_ORDER)
        seg_bvp_for_tpv = zscore_1d(seg_bvp_bp)
        tpv_full = extract_tpv_features(seg_bvp_for_tpv)

        window_counter += 1
        row = {
            "subject": subject_id,
            "window": f"window{window_counter}",
            "window_index": window_counter,
            "noise": noise_name,
            "alpha": float(alpha),
            "status": int(w["label"]),
            "status_name": "stress" if int(w["label"]) == 1 else "normal",
            "phase": w["phase"],
            "t_start_sec": t0,
            "t_end_sec": t1,
            "major_ratio": float(w["major_ratio"])
        }

        for idx in SELECTED_TPV_INDICES:
            row[f"TPV{idx}"] = float(tpv_full[idx])

        rows.append(row)

    return pd.DataFrame(rows)


def main():
    subject_dirs = sorted([d for d in glob.glob(os.path.join(ROOT_DIR, "S*")) if os.path.isdir(d)])
    all_dfs = []

    print(f"[INFO] Found {len(subject_dirs)} subject folders")

    for noise_name, alpha in NOISE_LEVELS.items():
        print(f"\n{'='*80}")
        print(f"[INFO] Noise level: {noise_name} (alpha={alpha})")
        print(f"{'='*80}")

        for subject_dir in subject_dirs:
            sid = os.path.basename(subject_dir)
            if sid in EXCLUDE_SUBJECTS:
                print(f"[INFO] Skip excluded subject: {sid}")
                continue

            seed_str = f"{sid}_{noise_name}_{RANDOM_SEED}"
            seed_val = int(hashlib.md5(seed_str.encode()).hexdigest()[:8], 16)
            rng = np.random.default_rng(seed_val)

            print(f"[INFO] Processing {sid} ...")
            try:
                df_sub = extract_subject_bvp_tpv_with_noise(
                    subject_dir=subject_dir,
                    noise_name=noise_name,
                    alpha=alpha,
                    rng=rng
                )
                print(f"[INFO] {sid}: extracted {len(df_sub)} windows")
                if len(df_sub) > 0:
                    all_dfs.append(df_sub)
            except Exception as e:
                print(f"[WARN] Failed on {sid} @ {noise_name}: {e}")

    if len(all_dfs) == 0:
        print("[WARN] No windows extracted.")
        return

    df_all = pd.concat(all_dfs, axis=0, ignore_index=True)

    base_cols = [
        "subject", "window", "window_index",
        "noise", "alpha",
        "status", "status_name", "phase",
        "t_start_sec", "t_end_sec", "major_ratio"
    ]
    feat_cols = [f"TPV{idx}" for idx in SELECTED_TPV_INDICES]

    final_cols = base_cols + feat_cols
    df_all = df_all[final_cols].copy()

    csv_all = os.path.join(OUTPUT_DIR, "StressPredict_BVP_TPV_noise_only_all.csv")
    df_all.to_csv(csv_all, index=False)

    for noise_name in NOISE_LEVELS.keys():
        df_noise = df_all[df_all["noise"] == noise_name].copy()
        save_path = os.path.join(OUTPUT_DIR, f"StressPredict_BVP_TPV_{noise_name}.csv")
        df_noise.to_csv(save_path, index=False)
        print(f"[INFO] Saved: {save_path}")

    summary = (
        df_all.groupby(["noise", "subject", "status_name"])
        .agg(n_windows=("window", "count"))
        .reset_index()
    )
    summary_csv = os.path.join(OUTPUT_DIR, "StressPredict_BVP_TPV_noise_only_summary.csv")
    summary.to_csv(summary_csv, index=False)

    print(f"[INFO] Saved: {csv_all}")
    print(f"[INFO] Saved: {summary_csv}")
    print("\n[INFO] Status counts")
    print(df_all.groupby(["noise", "status_name"]).size())


if __name__ == "__main__":
    main()

[INFO] Found 35 subject folders

[INFO] Noise level: alpha_0.01 (alpha=0.01)
[INFO] Processing S01 ...
[INFO] S01: extracted 44 windows
[INFO] Processing S02 ...
[INFO] S02: extracted 40 windows
[INFO] Processing S03 ...
[INFO] S03: extracted 38 windows
[INFO] Processing S04 ...
[INFO] S04: extracted 40 windows
[INFO] Processing S05 ...
[INFO] S05: extracted 25 windows
[INFO] Processing S06 ...
[INFO] S06: extracted 25 windows
[INFO] Processing S07 ...
[INFO] S07: extracted 26 windows
[INFO] Processing S08 ...
[INFO] S08: extracted 23 windows
[INFO] Processing S09 ...
[INFO] S09: extracted 25 windows
[INFO] Processing S10 ...
[INFO] S10: extracted 25 windows
[INFO] Processing S11 ...
[INFO] S11: extracted 25 windows
[INFO] Processing S12 ...
[INFO] S12: extracted 25 windows
[INFO] Processing S13 ...
[INFO] S13: extracted 26 windows
[INFO] Processing S14 ...
[INFO] S14: extracted 26 windows
[INFO] Processing S15 ...
[INFO] S15: extracted 28 windows
[INFO] Processing S16 ...
[INFO] S16: 

In [3]:
import os
import glob
import hashlib
import numpy as np
import pandas as pd

from scipy.signal import butter, filtfilt, find_peaks, welch
from scipy.interpolate import interp1d

# =============================================================================
# CONFIG
# =============================================================================
ROOT_DIR = r"/content/drive/MyDrive/Colab Notebooks/datasets/Stress-Predict-Dataset/Raw_data"
OUTPUT_DIR = r"/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Stress-Predict-Dataset/HRV-noise"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXCLUDE_SUBJECTS = {"S17"}

FS_BVP = 64.0

WINDOW_SEC = 60
STEP_SEC = 60
MAJORITY_RATIO_MIN = 0.95

# -----------------------------------------------------------------------------
# Noise settings
# -----------------------------------------------------------------------------
NOISE_LEVELS = {
    "alpha_0.01": 0.01,
    "alpha_0.03": 0.03,
    "alpha_0.05": 0.05,
}
RANDOM_SEED = 42

# BVP preprocessing
LOWCUT = 0.5
HIGHCUT = 8.0
FILTER_ORDER = 4

# Peak / IBI
MIN_HR_BPM = 40
MAX_HR_BPM = 180
MIN_PEAK_DISTANCE_SEC = 60.0 / MAX_HR_BPM

FS_IBI_INTERP = 4.0
LF_LOW = 0.04
LF_HIGH = 0.15
HF_LOW = 0.15
HF_HIGH = 0.40

IBI_MIN_SEC = 0.30
IBI_MAX_SEC = 1.50
IBI_DIFF_RATIO_MAX = 0.30


# =============================================================================
# Loaders
# =============================================================================
def load_bvp_csv(file_path, fs=FS_BVP):
    raw = pd.read_csv(file_path, header=None)
    start_ts = float(str(raw.iloc[0, 0]).strip())
    signal_fs = float(str(raw.iloc[1, 0]).strip())

    values = raw.iloc[2:, 0].astype(float).reset_index(drop=True).to_numpy(dtype=np.float64)
    timestamps = start_ts + np.arange(len(values), dtype=np.float64) / signal_fs

    return pd.DataFrame({
        "timestamp": timestamps,
        "bvp": values
    }), start_ts, signal_fs


def load_tags_csv(file_path, drop_first_tag=True):
    """
    tags_SXX.csv:
    - 각 행이 timestamp 하나
    - 일부 subject는 첫 tag가 실제 baseline 시작이 아니라
      더 이른 기록 시작 시점일 수 있으므로 기본적으로 제거
    """
    raw = pd.read_csv(file_path, header=None)

    tags = raw.iloc[:, 0].dropna().astype(float).reset_index(drop=True)

    if drop_first_tag:
        if len(tags) <= 1:
            raise ValueError(f"Not enough tags after dropping first tag: n={len(tags)}")
        tags = tags.iloc[1:].reset_index(drop=True)

    return pd.DataFrame({
        "tag_idx": np.arange(len(tags)),
        "timestamp": tags
    })


# =============================================================================
# Protocol intervals
# =============================================================================
def build_protocol_intervals(tags_df):
    """
    Stress-Predict protocol (after dropping the spurious first tag):
      t0~t1 : Baseline         -> 0
      t1~t2 : Stroop           -> 1
      t2~t3 : Rest1            -> 0
      t3~t4 : TSST             -> 1
      t4~t5 : Rest2            -> 0
      t5~t6 : Hyperventilation -> 1
      t6~t7 : Rest3            -> 0
      t7~t8 : Questionnaire    -> 0
    """
    timestamps = tags_df["timestamp"].values

    phase_info = [
        ("Baseline", 0),
        ("Stroop", 1),
        ("Rest1", 0),
        ("TSST", 1),
        ("Rest2", 0),
        ("Hyperventilation", 1),
        ("Rest3", 0),
        ("Questionnaire", 0),
    ]

    n_intervals_possible = len(timestamps) - 1
    n_use = min(n_intervals_possible, len(phase_info))

    if n_use <= 0:
        raise ValueError(f"No valid intervals from tags. n_tags={len(timestamps)}")

    rows = []
    for i in range(n_use):
        phase_name, label = phase_info[i]
        rows.append({
            "interval_idx": i,
            "phase": phase_name,
            "start": timestamps[i],
            "end": timestamps[i + 1],
            "label": label
        })

    return pd.DataFrame(rows)


def build_window_table_from_intervals(intervals_df, window_sec=60, step_sec=60, majority_ratio_min=0.95):
    """
    interval 내부에서만 window를 자르므로 major_ratio는 사실상 1.0
    """
    rows = []

    for _, r in intervals_df.iterrows():
        start = float(r["start"])
        end = float(r["end"])
        phase = r["phase"]
        label = int(r["label"])

        t = start
        while t + window_sec <= end:
            rows.append({
                "phase": phase,
                "label": label,
                "window_start": t,
                "window_end": t + window_sec,
                "major_ratio": 1.0
            })
            t += step_sec

    return pd.DataFrame(rows)


# =============================================================================
# Basic helpers
# =============================================================================
def fill_nan_with_median(x):
    x = np.asarray(x, dtype=np.float64).reshape(-1).copy()
    if np.isnan(x).any():
        med = np.nanmedian(x)
        if np.isnan(med):
            med = 0.0
        x = np.nan_to_num(x, nan=med, posinf=med, neginf=med)
    return x


def bandpass_filter(sig, fs=64, low=0.5, high=8.0, order=4):
    sig = np.asarray(sig, dtype=np.float64).reshape(-1)
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, sig).astype(np.float64)


def zscore_1d(x):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    mu = float(np.mean(x))
    sd = float(np.std(x))
    if sd < 1e-8:
        return np.zeros_like(x, dtype=np.float64)
    return ((x - mu) / (sd + 1e-8)).astype(np.float64)


def safe_div(a, b):
    return float(a / b) if abs(b) > 1e-12 else np.nan


def add_gaussian_noise(sig, alpha, rng):
    sig = np.asarray(sig, dtype=np.float64).reshape(-1)
    if alpha <= 0:
        return sig.copy()

    sig_std = float(np.std(sig))
    if sig_std < 1e-12:
        return sig.copy()

    noise = rng.normal(loc=0.0, scale=alpha * sig_std, size=len(sig)).astype(np.float64)
    return (sig + noise).astype(np.float64)


# =============================================================================
# HRV helpers
# =============================================================================
def detect_ppg_peaks(sig, fs):
    sig = np.asarray(sig, dtype=np.float64).reshape(-1)
    distance = max(1, int(round(MIN_PEAK_DISTANCE_SEC * fs)))
    prom = max(0.10, 0.20 * float(np.std(sig)))
    peaks, props = find_peaks(sig, distance=distance, prominence=prom)
    return peaks, props


def build_filtered_ibi_series(ibi_times_abs, ibi_values):
    ibi_times_abs = np.asarray(ibi_times_abs, dtype=np.float64).reshape(-1)
    ibi_values = np.asarray(ibi_values, dtype=np.float64).reshape(-1)

    if len(ibi_times_abs) < 2 or len(ibi_values) < 2:
        return None

    plausible = (ibi_values >= IBI_MIN_SEC) & (ibi_values <= IBI_MAX_SEC)
    ibi_values = ibi_values[plausible]
    ibi_times_abs = ibi_times_abs[plausible]

    if len(ibi_values) < 2:
        return None

    med_ibi = float(np.median(ibi_values))
    stable = np.abs(ibi_values - med_ibi) <= (IBI_DIFF_RATIO_MAX * (med_ibi + 1e-8))

    ibi_values = ibi_values[stable]
    ibi_times_abs = ibi_times_abs[stable]

    if len(ibi_values) < 2:
        return None

    return {
        "ibi_times_abs": ibi_times_abs,
        "ibi": ibi_values,
        "ibi_median_sec": med_ibi
    }


def interpolate_ibi(ibi_times_abs, ibi, fs_interp=FS_IBI_INTERP):
    ibi_times_abs = np.asarray(ibi_times_abs, dtype=np.float64).reshape(-1)
    ibi = np.asarray(ibi, dtype=np.float64).reshape(-1)

    if len(ibi_times_abs) < 2 or len(ibi) < 2:
        return None
    if ibi_times_abs[-1] <= ibi_times_abs[0]:
        return None

    t_uniform = np.arange(ibi_times_abs[0], ibi_times_abs[-1], 1.0 / fs_interp)
    if len(t_uniform) < 4:
        return None

    try:
        f_interp = interp1d(
            ibi_times_abs,
            ibi,
            kind="linear",
            bounds_error=False,
            fill_value="extrapolate",
            assume_sorted=True
        )
        ibi_uniform = f_interp(t_uniform)
    except Exception:
        return None

    ibi_uniform = np.asarray(ibi_uniform, dtype=np.float64)
    if np.any(~np.isfinite(ibi_uniform)):
        return None

    return {"t_uniform": t_uniform, "ibi_uniform": ibi_uniform}


def compute_time_domain_hrv(ibi):
    ibi = np.asarray(ibi, dtype=np.float64).reshape(-1)

    if len(ibi) < 2:
        return {
            "RMSSD": np.nan,
            "SDNN": np.nan,
            "ibi_mean_sec": np.nan,
            "ibi_std_sec": np.nan,
        }

    diff_ibi = np.diff(ibi)
    rmssd = np.sqrt(np.mean(diff_ibi ** 2)) if len(diff_ibi) > 0 else np.nan
    sdnn = np.std(ibi, ddof=0)

    return {
        "RMSSD": float(rmssd) if np.isfinite(rmssd) else np.nan,
        "SDNN": float(sdnn) if np.isfinite(sdnn) else np.nan,
        "ibi_mean_sec": float(np.mean(ibi)) if np.isfinite(np.mean(ibi)) else np.nan,
        "ibi_std_sec": float(np.std(ibi, ddof=0)) if np.isfinite(np.std(ibi, ddof=0)) else np.nan,
    }


def compute_freq_domain_hrv(ibi_uniform, fs_interp=FS_IBI_INTERP):
    ibi_uniform = np.asarray(ibi_uniform, dtype=np.float64).reshape(-1)
    if len(ibi_uniform) < 8:
        return {"LF": np.nan, "HF": np.nan, "LF_HF": np.nan}

    nperseg = min(256, len(ibi_uniform))
    f, pxx = welch(ibi_uniform, fs=fs_interp, nperseg=nperseg)

    lf_mask = (f >= LF_LOW) & (f < LF_HIGH)
    hf_mask = (f >= HF_LOW) & (f < HF_HIGH)

    lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
    hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan
    lf_hf = safe_div(lf, hf) if np.isfinite(lf) and np.isfinite(hf) else np.nan

    return {
        "LF": float(lf) if np.isfinite(lf) else np.nan,
        "HF": float(hf) if np.isfinite(hf) else np.nan,
        "LF_HF": float(lf_hf) if np.isfinite(lf_hf) else np.nan,
    }


def extract_hrv_features_from_bvp_segment(seg_bvp, fs_bvp=FS_BVP):
    seg_bvp = fill_nan_with_median(seg_bvp)
    seg_bvp_bp = bandpass_filter(seg_bvp, fs=fs_bvp, low=LOWCUT, high=HIGHCUT, order=FILTER_ORDER)
    seg_bvp_z = zscore_1d(seg_bvp_bp)

    peaks, _ = detect_ppg_peaks(seg_bvp_z, fs=fs_bvp)

    row = {
        "n_peaks": int(len(peaks)),
        "n_ibi": 0,
        "ibi_pass_ratio": np.nan,
        "valid_ibi_count": 0,
        "ibi_mean_sec": np.nan,
        "ibi_std_sec": np.nan,
        "RMSSD": np.nan,
        "SDNN": np.nan,
        "LF": np.nan,
        "HF": np.nan,
        "LF_HF": np.nan,
    }

    if len(peaks) < 2:
        return row

    peak_times = peaks.astype(np.float64) / float(fs_bvp)
    ibi_values = np.diff(peak_times)
    ibi_times_abs = peak_times[1:]

    row["n_ibi"] = int(len(ibi_values))

    plausible = (ibi_values >= IBI_MIN_SEC) & (ibi_values <= IBI_MAX_SEC)
    row["ibi_pass_ratio"] = float(np.mean(plausible)) if len(ibi_values) > 0 else np.nan

    pack = build_filtered_ibi_series(ibi_times_abs, ibi_values)
    if pack is None:
        return row

    ibi = pack["ibi"]
    ibi_times_abs = pack["ibi_times_abs"]
    row["valid_ibi_count"] = int(len(ibi))

    td = compute_time_domain_hrv(ibi)
    row.update(td)

    interp_pack = interpolate_ibi(ibi_times_abs, ibi, fs_interp=FS_IBI_INTERP)
    if interp_pack is None:
        return row

    fd = compute_freq_domain_hrv(interp_pack["ibi_uniform"], fs_interp=FS_IBI_INTERP)
    row.update(fd)
    return row


# =============================================================================
# Main extraction (noise only)
# =============================================================================
def extract_subject_hrv_with_noise(subject_dir, noise_name, alpha, rng):
    subject_id = os.path.basename(subject_dir)
    if subject_id in EXCLUDE_SUBJECTS:
        return pd.DataFrame()

    bvp_path = os.path.join(subject_dir, "BVP.csv")
    tags_path = os.path.join(subject_dir, f"tags_{subject_id}.csv")

    if not (os.path.exists(bvp_path) and os.path.exists(tags_path)):
        return pd.DataFrame()

    bvp_df, _, fs_bvp = load_bvp_csv(bvp_path, fs=FS_BVP)
    tags_df = load_tags_csv(tags_path, drop_first_tag=True)

    if len(tags_df) < 2:
        print(f"[WARN] {subject_id}: too few tags after dropping first tag")
        return pd.DataFrame()

    if int(round(fs_bvp)) != int(FS_BVP):
        print(f"[WARN] {subject_id}: fs mismatch BVP={fs_bvp}")

    intervals_df = build_protocol_intervals(tags_df)
    window_df = build_window_table_from_intervals(
        intervals_df,
        WINDOW_SEC,
        STEP_SEC,
        MAJORITY_RATIO_MIN
    )

    bvp_raw = bvp_df["bvp"].values.astype(np.float64)
    bvp_raw = fill_nan_with_median(bvp_raw)
    bvp_noisy = add_gaussian_noise(bvp_raw, alpha=alpha, rng=rng)

    bvp_time = bvp_df["timestamp"].values.astype(np.float64)

    rows = []
    window_counter = 0
    expected_len = int(WINDOW_SEC * FS_BVP)

    for _, w in window_df.iterrows():
        t0 = float(w["window_start"])
        t1 = float(w["window_end"])

        mask_bvp = (bvp_time >= t0) & (bvp_time < t1)
        seg_bvp = bvp_noisy[mask_bvp]

        if len(seg_bvp) != expected_len:
            continue

        hrv_info = extract_hrv_features_from_bvp_segment(seg_bvp, fs_bvp=FS_BVP)

        window_counter += 1
        row = {
            "subject": subject_id,
            "window": f"window{window_counter}",
            "window_index": window_counter,
            "window_sec": WINDOW_SEC,
            "noise": noise_name,
            "alpha": float(alpha),
            "status": int(w["label"]),
            "status_name": "stress" if int(w["label"]) == 1 else "normal",
            "phase": w["phase"],
            "t_start_sec": t0,
            "t_end_sec": t1,
            "major_ratio": float(w["major_ratio"])
        }
        row.update(hrv_info)
        rows.append(row)

    return pd.DataFrame(rows)


def main():
    subject_dirs = sorted([d for d in glob.glob(os.path.join(ROOT_DIR, "S*")) if os.path.isdir(d)])
    all_dfs = []

    print(f"[INFO] Found {len(subject_dirs)} subject folders")

    for noise_name, alpha in NOISE_LEVELS.items():
        print(f"\n{'='*80}")
        print(f"[INFO] Noise level: {noise_name} (alpha={alpha})")
        print(f"{'='*80}")

        for subject_dir in subject_dirs:
            sid = os.path.basename(subject_dir)
            if sid in EXCLUDE_SUBJECTS:
                print(f"[INFO] Skip excluded subject: {sid}")
                continue

            seed_str = f"{sid}_{noise_name}_{RANDOM_SEED}"
            seed_val = int(hashlib.md5(seed_str.encode()).hexdigest()[:8], 16)
            rng = np.random.default_rng(seed_val)

            print(f"[INFO] Processing {sid} ...")
            try:
                df_sub = extract_subject_hrv_with_noise(
                    subject_dir=subject_dir,
                    noise_name=noise_name,
                    alpha=alpha,
                    rng=rng
                )
                print(f"[INFO] {sid}: extracted {len(df_sub)} windows")
                if len(df_sub) > 0:
                    all_dfs.append(df_sub)
            except Exception as e:
                print(f"[WARN] Failed on {sid} @ {noise_name}: {e}")

    if len(all_dfs) == 0:
        print("[WARN] No windows extracted.")
        return

    df_all = pd.concat(all_dfs, axis=0, ignore_index=True)

    base_cols = [
        "subject", "window", "window_index", "window_sec",
        "noise", "alpha",
        "status", "status_name", "phase",
        "t_start_sec", "t_end_sec", "major_ratio"
    ]
    hrv_cols = [
        "n_peaks",
        "n_ibi",
        "ibi_pass_ratio",
        "valid_ibi_count",
        "ibi_mean_sec",
        "ibi_std_sec",
        "RMSSD",
        "SDNN",
        "LF",
        "HF",
        "LF_HF",
    ]

    df_all = df_all[base_cols + hrv_cols].copy()

    csv_path = os.path.join(OUTPUT_DIR, "StressPredict_BVP_HRV_1min_noise_only_all.csv")
    df_all.to_csv(csv_path, index=False)

    for noise_name in NOISE_LEVELS.keys():
        df_noise = df_all[df_all["noise"] == noise_name].copy()
        save_path = os.path.join(OUTPUT_DIR, f"StressPredict_BVP_HRV_1min_{noise_name}.csv")
        df_noise.to_csv(save_path, index=False)
        print(f"[INFO] Saved: {save_path}")

    summary = (
        df_all.groupby(["noise", "subject", "status_name"])
        .agg(
            n_windows=("window", "count"),
            n_valid_rmssd=("RMSSD", lambda x: int(np.sum(np.isfinite(x)))),
            n_valid_lf_hf=("LF_HF", lambda x: int(np.sum(np.isfinite(x)))),
            mean_n_peaks=("n_peaks", "mean"),
            mean_n_ibi=("n_ibi", "mean"),
            mean_valid_ibi=("valid_ibi_count", "mean"),
        )
        .reset_index()
    )
    summary_csv = os.path.join(OUTPUT_DIR, "StressPredict_BVP_HRV_1min_noise_only_summary.csv")
    summary.to_csv(summary_csv, index=False)

    print(f"[INFO] Saved: {csv_path}")
    print(f"[INFO] Saved: {summary_csv}")
    print("\n[INFO] Status counts")
    print(df_all.groupby(["noise", "status_name"]).size())


if __name__ == "__main__":
    main()

[INFO] Found 35 subject folders

[INFO] Noise level: alpha_0.01 (alpha=0.01)
[INFO] Processing S01 ...
[INFO] S01: extracted 44 windows
[INFO] Processing S02 ...


/tmp/ipykernel_11887/96700201.py:325: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
/tmp/ipykernel_11887/96700201.py:326: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan
/tmp/ipykernel_11887/96700201.py:325: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
/tmp/ipykernel_11887/96700201.py:326: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan


[INFO] S02: extracted 40 windows
[INFO] Processing S03 ...
[INFO] S03: extracted 38 windows
[INFO] Processing S04 ...


/tmp/ipykernel_11887/96700201.py:325: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
/tmp/ipykernel_11887/96700201.py:326: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan
/tmp/ipykernel_11887/96700201.py:325: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
/tmp/ipykernel_11887/96700201.py:326: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan


[INFO] S04: extracted 40 windows
[INFO] Processing S05 ...
[INFO] S05: extracted 25 windows
[INFO] Processing S06 ...
[INFO] S06: extracted 25 windows
[INFO] Processing S07 ...


...

[INFO] S29: extracted 24 windows
[INFO] Processing S30 ...
[INFO] S30: extracted 25 windows
[INFO] Processing S31 ...
[INFO] S31: extracted 24 windows
[INFO] Processing S32 ...


/tmp/ipykernel_11887/96700201.py:325: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
/tmp/ipykernel_11887/96700201.py:326: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan
/tmp/ipykernel_11887/96700201.py:325: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
/tmp/ipykernel_11887/96700201.py:326: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan
/tmp/ipykern

[INFO] S32: extracted 25 windows
[INFO] Processing S33 ...
[INFO] S33: extracted 24 windows
[INFO] Processing S34 ...
[INFO] S34: extracted 25 windows
[INFO] Processing S35 ...


/tmp/ipykernel_11887/96700201.py:325: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
/tmp/ipykernel_11887/96700201.py:326: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan
/tmp/ipykernel_11887/96700201.py:325: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
/tmp/ipykernel_11887/96700201.py:326: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan
/tmp/ipykern

[INFO] S35: extracted 25 windows
[INFO] Saved: /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Stress-Predict-Dataset/HRV-noise/StressPredict_BVP_HRV_1min_alpha_0.01.csv
[INFO] Saved: /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Stress-Predict-Dataset/HRV-noise/StressPredict_BVP_HRV_1min_alpha_0.03.csv
[INFO] Saved: /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Stress-Predict-Dataset/HRV-noise/StressPredict_BVP_HRV_1min_alpha_0.05.csv
[INFO] Saved: /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Stress-Predict-Dataset/HRV-noise/StressPredict_BVP_HRV_1min_noise_only_all.csv
[INFO] Saved: /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Stress-Predict-Dataset/HRV-noise/StressPredict_BVP_HRV_1min_noise_only_summary.csv

[INFO] Status counts
noise       status_name
alpha_0.01  normal         520
            stress         390
alpha_0.03  normal         520
            stress         390
alpha_0.05  normal         520
            stress         390
dtype: int64
